In [ ]:
!pip install accelerate==1.10.0
!pip install datasets==4.0.0
!pip install peft==0.17.0
!pip install transformers==4.55.2
!pip install trl==0.21.0
!pip install mistral-common==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.10.1
    Uninstalling accelerate-1.10.1:
      Successfully uninstalled accelerate-1.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 34.3 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.17.1
    Uninstalling peft-0.17.1:
      Successfully uninstalled peft-0.17.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 160.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 128.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling tran

In [ ]:
import accelerate
import datasets
import peft
import transformers
import trl

print("Library Versions:")
print(f"accelerate     : {accelerate.__version__}")
print(f"datasets       : {datasets.__version__}")
print(f"peft           : {peft.__version__}")
print(f"transformers   : {transformers.__version__}")
print(f"trl            : {trl.__version__}")
print(f"mistral-common : {trl.__version__}")

Library Versions:
accelerate     : 1.10.0
datasets       : 4.0.0
peft           : 0.17.0
transformers   : 4.55.2
trl            : 0.21.0
mistral-common : 0.21.0


In [ ]:
# Verify PyTorch and CUDA versions for compatibility with the training setup.
# Used in latest run.
# PyTorch 2.8.0+cu126 built against CUDA 12.6.
# NVIDIA A100 (Driver 550.54.15, CUDA 12.4 runtime).
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)

PyTorch Version: 2.8.0+cu126
CUDA Version: 12.6


In [ ]:
# Pick a valid device index (default to 0 if there's only one GPU)
num_gpus = torch.cuda.device_count()
device_id = 1 if num_gpus > 1 else 0
device = f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"

# Load the base model (full precision fp16 so merging is supported)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",  # Let HF place layers; avoid manual .to(...) afterward
)

# Load the LoRA weights (attach to base)
lora_model = PeftModel.from_pretrained(base_model, new_model)

# IMPORTANT: Do NOT manually move when using device_map="auto"
# lora_model.to(device)  # <-- remove to avoid conflicts with device_map

# Merge the LoRA weights with the base model weights (produces a plain AutoModelForCausalLM)
model = lora_model.merge_and_unload()

# Again, avoid fighting device_map placement; only move if you're on CPU-only
# model.to(device)  # <-- not needed when device_map="auto"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Import text generation pipeline
from transformers import pipeline

# Get fine-tuned model from trainer
gen_model = trainer.model
gen_model.eval()

# Disable gradient checkpointing if active
try:
    gen_model.gradient_checkpointing_disable()
except Exception:
    pass

# Enable cache for faster generation
gen_model.config.use_cache = True

# Set tokenizer padding configuration
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Define user question for model prompt
question = "How can I write a Python program that calculates the mean, standard " "deviation and coefficient of variation of a dataset from a CSV file?"
messages = [
    {"role": "system", "content": "You are a helpful Python tutor."},
    {"role": "user", "content": question},
]

# Build model input prompt safely
try:
    # Use chat template if tokenizer supports it
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    # Fallback to Llama-style system prompt
    bos = tokenizer.bos_token or ""
    sys_prompt = "You are a helpful Python tutor."
    prompt = f"{bos}[INST] <<SYS>>\n{sys_prompt}\n<</SYS>>\n\n{question} [/INST]"

# Create text generation pipeline
pipe = pipeline(
    task="text-generation",
    model=gen_model,
    tokenizer=tokenizer,
    device_map="auto",
)

# Generate model response text
out = pipe(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=6,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

# Display generated answer
print(out[0]["generated_text"])

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 To calculate the mean, standard deviation and coefficient of variation of the dataset from a CSV file, you can use the following code:

```python
import pandas as pd

# Load the CSV file
df = pd.read_csv('data.csv')

# Calculate the mean
mean = df['value'].mean()

# Calculate standard deviation
std = df['value'].std()

# Calculated the coefficient of variation
cv = (std / mean) * 100

print("Mean:", mean)
print("Standard Deviation:", std)
print("Coefficient of Variation:", cv)
```

In this code, we first import the pandas library to read the CSV file. Then, we load the CSV file using the `pd.read_csv()` function. The `mean` variable is calculated by calling the `mean()` method on the `df['value']` series. Similarly, the `std` variable is calculated by using the `std()` method on the same series. Finally, the `cv` variable is calculated by dividing the standard deviation by the mean and multiplying it by 100. The results are then printed.


In [ ]:
# Check the number of GPUs available
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

# Check if CUDA device 1 is available
if num_gpus > 1:
    print("cuda:1 is available.")
else:
    print("cuda:1 is not available.")

Number of GPUs available: 1
cuda:1 is not available.


In [ ]:
!nvidia-smi

Tue Oct 21 23:35:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       5MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Import the libraries and modules used for fine-tuning a large language model (LLM).

In [ ]:
# Access OS utilities, system commands and process management tools
import os, sys, shutil, subprocess

# Work with filesystem paths in a cross-platform, object-oriented way
from pathlib import Path

# Google drive mounting capability
from google.colab import drive

# Date time from system
from datetime import datetime

# Load and manage training datasets from the Hugging Face Hub
from datasets import load_dataset

# Import model, tokenizer and training utilities from the Transformers library
from transformers import (
    AutoModelForCausalLM,  # Loads a causal language model (e.g., Llama)
    AutoTokenizer,  # Handles text tokenization for the model
    HfArgumentParser,  # Parses command-line or script arguments
    TrainingArguments,  # Defines training configurations and hyperparameters
    pipeline,  # Creates ready-to-use NLP pipelines for inference
    logging,  # Controls logging verbosity and output
)

# Parameter-Efficient Fine-Tuning (PEFT)
# Give the ability to adapt LLMs without having to redo all the weights.
# Utilizes Low-Rank Adapter (LoRA).
# Efficiently fine-tune without redoing all their parameters.
# Overall reduces compute cost while preserving performance of the model.
from peft import LoraConfig, PeftModel, get_peft_model

# Transformers Reinforcement Learning (TRL)
# Supervised Fine-Tuning (SFT)
# Import the SFTTrainer to manage supervised fine-tuning
from trl import SFTTrainer

# Define and manage training configuration settings (e.g., epochs, batch size, learning rate)
from transformers import TrainingArguments

# Import model and tokenizer classes for causal language modeling
from transformers import AutoModelForCausalLM, AutoTokenizer

# Import Hugging Face pipeline for streamlined text generation tasks
from transformers import pipeline

# Import login utility to authenticate with the Hugging Face Hub
from huggingface_hub import login

# Securely prompt for the API key without showing it in the terminal
import getpass

# Environment Variables

In [ ]:
# If your notebook sometimes hides devices, force the first GPU visible:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Disable Weights & Biases tracking to prevent automatic logging
os.environ["WANDB_DISABLED"] = "true"

# Diagnostics for invalid arguments in transformers pipeline
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

# Hugging Face Authentication

In [ ]:
# Prompt the user to enter their Hugging Face API key securely (hidden input)
hugging_face_key = getpass.getpass("Enter your Hugging Face token: ")

# Log in to Hugging Face Hub using the entered key
login(hugging_face_key)

Enter your Hugging Face token: ··········


# Dataset for fine-tuning

In [ ]:
# Dataset for fine-tuning
dataset_name = "nikhiljatiwal/minipython-Alpaca-14k"

dataset = load_dataset(dataset_name)

# Get the dataset structure and size
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/387 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/25.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 14000
    })
})


# Model input and output config.

In [ ]:
# Hugging Face model to train
model_name = "NousResearch/llama-2-7b-chat-hf"  # Optimized for the chat bot.

# Output model name
new_model = "/kaggle/working/llama-2-7b-codeAlpaca"

# Quantized + LoRA = QLoRA which is a method of fine-tuning LLMs more cheaply and efficiently without sacrificing accuracy.

In [ ]:
# LoRA attention dimension (rank)
# The higher the value the more fine-grained updates at the cost of increased
# memory usage
lora_r = 64

# LoRA alpha (scaling factor)
# How much the LoRA layes change the original LLM's behavior.
# Higher value = updates stronger : Lower value = keeps changes smaller.
lora_alpha = 16

# LoRA dropout probability
# Randomly turns off some connections during training to avoid overfitting.
lora_dropout = 0.1

# Training Arguments parameters

In [ ]:
# Output directory where the model predictions and checkpoints will be stored
output_dir = "/kaggle/working/llama-2-7b-codeAlpaca"

# Number of training epochs
num_train_epochs = 1

# Enable fp16 training (set to True for mixed precision training)
fp16 = True

# Batch size per GPU for training
per_device_train_batch_size = 8

# Batch size per GPU for evaluation
per_device_eval_batch_size = 8

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 2

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient norm (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "adamw_torch"

# Learning rate schedule
lr_scheduler_type = "constant"

# Group sequences into batches with the same length
# Saves memory and speeds up training considerably
group_by_length = True

# Ratio of steps for a linear warmup
warmup_ratio = 0.03

# Save checkpoint every X updates steps
save_steps = 100

# Log every X updates steps
logging_steps = 10

# Supervised Fine-Tuning (SFT) Model and Tokenizer Setup

In [ ]:
# Set max token length for each training example
max_seq_length = None

# Combine shorter samples for faster training
packing = False

# Load the training dataset
dataset = load_dataset(dataset_name, split="train")

# Load and configure the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the base model in 8-bit precision for memory efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Enable gradient checkpointing and input gradients for fine-tuning
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

# LoRA Configuration, Dataset Tokenization and Trainer Initialization

In [ ]:
# Configure and apply LoRA to the base model
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Define training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    max_grad_norm=max_grad_norm,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    fp16=fp16,
    group_by_length=True,
)


# Tokenize dataset and add padding for consistent lengths.
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,  # limit sequence length
    )


tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Initialize the supervised fine-tuning trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14000 [00:00<?, ? examples/s]

# Model Training

In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.597500
20,0.971200
30,0.898600
40,0.804500
50,0.846100
60,0.782300
70,0.752000
80,0.754100
90,0.712500
100,0.769600


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

# Generate and Display Model Response with Chat Prompt

In [ ]:
# Import text generation pipeline
from transformers import pipeline

# Get fine-tuned model from trainer
gen_model = trainer.model
gen_model.eval()

# Disable gradient checkpointing if active
try:
    gen_model.gradient_checkpointing_disable()
except Exception:
    pass

# Enable cache for faster generation
gen_model.config.use_cache = True

# Set tokenizer padding configuration
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Define user question for model prompt
question = "How can I write a Python program that calculates the mean, standard " "deviation and coefficient of variation of a dataset from a CSV file?"
messages = [
    {"role": "system", "content": "You are a helpful Python tutor."},
    {"role": "user", "content": question},
]

# Build model input prompt safely
try:
    # Use chat template if tokenizer supports it
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    # Fallback to Llama-style system prompt
    bos = tokenizer.bos_token or ""
    sys_prompt = "You are a helpful Python tutor."
    prompt = f"{bos}[INST] <<SYS>>\n{sys_prompt}\n<</SYS>>\n\n{question} [/INST]"

# Create text generation pipeline
pipe = pipeline(
    task="text-generation",
    model=gen_model,
    tokenizer=tokenizer,
    device_map="auto",
)

# Generate model response text
out = pipe(
    prompt,
    max_new_tokens=300,
    repetition_penalty=1.05,
    no_repeat_ngram_size=6,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

# Display generated answer
print(out[0]["generated_text"])

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 To calculate the mean, standard deviation and coefficient of variation of the dataset from a CSV file, you can use the following code:

```python
import pandas as pd

# Load the CSV file
df = pd.read_csv('data.csv')

# Calculate the mean
mean = df['value'].mean()

# Calculate standard deviation
std = df['value'].std()

# Calculated the coefficient of variation
cv = (std / mean) * 100

print("Mean:", mean)
print("Standard Deviation:", std)
print("Coefficient of Variation:", cv)
```

In this code, we first import the pandas library to read the CSV file. Then, we load the CSV file using the `pd.read_csv()` function. The `mean` variable is calculated by calling the `mean()` method on the `df['value']` series. Similarly, the `std` variable is calculated by using the `std()` method on the same series. Finally, the `cv` variable is calculated by dividing the standard deviation by the mean and multiplying it by 100. The results are then printed.


# Load Base Model, Merge LoRA Weights and Prepare Tokenizer

In [ ]:
# Pick a valid device index (default to 0 if there's only one GPU)
num_gpus = torch.cuda.device_count()
device_id = 1 if num_gpus > 1 else 0
device = f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,  # so merging is supported
    device_map="auto",  # Let Hugging Face Transformers to place layers
)

# Load the LoRA weights (attach to base)
lora_model = PeftModel.from_pretrained(base_model, new_model)

# IMPORTANT!: Do NOT manually move when using device_map="auto"
# lora_model.to(device)

# Merge the LoRA weights with the base model weights
# Produces a plain AutoModelForCausalLM
merged_model = lora_model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained("/kaggle/working/llama-2-7b-codeAlpaca-merged")

# IMPORTANT!: Only move if you're on CPU-only
# model.to(device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Merged Models Test

In [ ]:
# Use merged model for text generation
gen_model = model
gen_model.eval()

# Disable gradient checkpointing for inference
try:
    gen_model.gradient_checkpointing_disable()
except Exception:
    pass

# Enable cache for faster inference
gen_model.config.use_cache = True

# Set tokenizer padding and alignment
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Define user question and conversation messages
question = "How can I write a Python program that calculates the mean, standard " "deviation and coefficient of variation of a dataset from a CSV file?"
messages = [
    {"role": "system", "content": "You are a helpful Python tutor."},
    {"role": "user", "content": question},
]

# Build model prompt safely and consistently
try:
    # Use tokenizer chat template if available
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
except Exception:
    # Fallback to simple Llama-style instruction prompt
    bos = tokenizer.bos_token or ""
    sys_prompt = "You are a helpful Python tutor."
    prompt = f"{bos}[INST] <<SYS>>\n{sys_prompt}\n<</SYS>>\n\n{question} [/INST]"

# Create generation pipeline for inference
pipe = pipeline(
    task="text-generation",
    model=gen_model,
    tokenizer=tokenizer,
    device_map="auto",
)

# Generate and display model response
out = pipe(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=6,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

# Print generated answer text
print(out[0]["generated_text"])

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 To calculate the mean, standard deviation, and coefficient of variation of the dataset from a CSV file, you can use the following code:

```python
import pandas as pd

# Load the CSV file
df = pd.read_csv('data.csv')

# Calculate the mean
mean = df['value'].mean()

# Calculate standard deviation
std = df['value'].std()

# Calculated the coefficient of variation
cv = (std / mean) * 100

print("Mean:", mean)
print("Standard Deviation:", std)
print("Coefficient of Variation:", cv)
```

In this code, we first import the pandas library to read the CSV file. Then, we load the CSV file using the `pd.read_csv()` function. The `mean` variable is calculated by calling the `mean()` method on the `df['value']` series. Similarly, the `std` variable is calculated by using the `std()` method on the same series. Finally, the `cv` variable is calculated by dividing the standard deviation by the mean and multiplying it by 100. The results are then printed.


# Explicit Code Rewrite Example (testing code writing capability)

In [ ]:
# Prepare merged model for secure code generation
gen_model = merged_model  # merged model
gen_model.eval()
try:
    gen_model.gradient_checkpointing_disable()
except Exception:
    pass
gen_model.config.use_cache = True

# Configure tokenizer padding and alignment
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Define system role and behavior instructions
system_prompt = (
    "You are a senior Python security engineer and tutor. Your job is to review code, identify "
    "vulnerabilities in plain English and provide concise, production-ready fixes. Favor secure "
    "defaults, minimal dependencies and clear comments. Never provide exploit payloads or step-by-step "
    "attack instructions; focus on remediation, safe patterns and brief justifications. When asked to "
    "rewrite code, preserve functionality where possible, improve input validation add minimal tests, "
    "and document trade-offs."
)

# Define vulnerable code sample for review
code_snippet = """import pickle

def load_user_data(serialized_data):
    \"""
    Deserializes user data from a byte stream.
    ...
    \"""
    user_data = pickle.loads(serialized_data)
    return user_data

if __name__ == "__main__":
    malicious_input = input("Enter serialized data: ")
    try:
        serialized_data = bytes.fromhex(malicious_input)
        load_user_data(serialized_data)
    except Exception as e:
        print(f"Deserialization failed: {e}")"""

# Define user prompt with review and rewrite instructions
user_prompt = (
    "Please review the following script and do all of the following in order:\n"
    "1) Identify the primary vulnerability and why it’s dangerous.\n"
    "2) Explain realistic impact and when it can be exploited.\n"
    "3) Rewrite the program securely **without `pickle`** (e.g., use `json` + validation) while keeping the same basic CLI behavior.\n"
    "4) Add a tiny unit test snippet that demonstrates safe behavior (no network/files).\n"
    "5) Briefly list safer serialization options and when to use them.\n"
    "6) Keep the answer concise and do **not** include exploit examples.\n\n"
    f"Here is the code to analyze and refactor:\n\n```python\n{code_snippet}\n```"
)

# Create structured message list for chat-style generation
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# Build prompt safely for chat model
try:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
except Exception:
    bos = tokenizer.bos_token or ""
    prompt = f"{bos}[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_prompt} [/INST]"

# Create text generation pipeline
pipe = pipeline(
    task="text-generation",
    model=gen_model,
    tokenizer=tokenizer,
    device_map="auto",
)

# Generate secure refactoring and review output
with torch.inference_mode():
    out = pipe(
        prompt,
        max_new_tokens=1200,  # give the model room to finish
        do_sample=True,  # make temperature/top_p valid (no warning)
        temperature=0.2,  # conservative sampling for stability
        top_p=0.9,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        return_full_text=False,
    )

# Display model-generated secure code review
print(out[0]["generated_text"])

Device set to use cuda:0


 ```python
import json

def load_data(data):
    """Deserializes data from a string."""
    try:
       data = json.loads(data)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format: {e}")
    return data

if __name__. == "__main__"
    data = input("Enter serialised data: ")
    data = bytes.fromhex(''.join(map(chr, data)))
    try:
         user_data = load_data(data)
    finally:
        print(user_data)
```

This code is more secure than the original because it uses `json` for deserialization instead of `pickle`. `json` is a standard library and has built-in validation, which makes it harder to inject malicious data. It also avoids the potential security risks associated with `pickle`, such as the ability to manipulate the deserialized data.

The code also includes a simple unit test to demonstrate the safe behavior of the function. The test checks that the function correctly deserializes valid JSON data and raises an error if the input is not a valid

# Get current working directory

In [ ]:
!pwd

/content


# List out workspace for exported model that is fine-tuned

In [ ]:
!ls -l /kaggle/working/llama-2-7b-codeAlpaca/

total 131144
-rw-r--r-- 1 root root       865 Oct 22 00:11 adapter_config.json
-rw-r--r-- 1 root root 134235048 Oct 22 00:11 adapter_model.safetensors
drwxr-xr-x 2 root root      4096 Oct 21 23:43 checkpoint-100
drwxr-xr-x 2 root root      4096 Oct 21 23:47 checkpoint-200
drwxr-xr-x 2 root root      4096 Oct 21 23:51 checkpoint-300
drwxr-xr-x 2 root root      4096 Oct 21 23:54 checkpoint-400
drwxr-xr-x 2 root root      4096 Oct 21 23:58 checkpoint-500
drwxr-xr-x 2 root root      4096 Oct 22 00:01 checkpoint-600
drwxr-xr-x 2 root root      4096 Oct 22 00:05 checkpoint-700
drwxr-xr-x 2 root root      4096 Oct 22 00:09 checkpoint-800
drwxr-xr-x 2 root root      4096 Oct 22 00:11 checkpoint-875
-rw-r--r-- 1 root root      1569 Oct 22 00:11 README.md
drwxr-xr-x 3 root root      4096 Oct 21 23:40 runs


# List out workspace for merged model

In [ ]:
!ls -l /kaggle/working/llama-2-7b-codeAlpaca-merged

total 13161052
-rw-r--r-- 1 root root        698 Oct 22 00:50 config.json
-rw-r--r-- 1 root root        195 Oct 22 00:50 generation_config.json
-rw-r--r-- 1 root root 4938985248 Oct 22 00:50 model-00001-of-00003.safetensors
-rw-r--r-- 1 root root 4947390768 Oct 22 00:50 model-00002-of-00003.safetensors
-rw-r--r-- 1 root root 3590488736 Oct 22 00:51 model-00003-of-00003.safetensors
-rw-r--r-- 1 root root      23986 Oct 22 00:51 model.safetensors.index.json


# Configure Llama CPP

In [ ]:
REPO = "/content/llama.cpp"

# Clone llama.cpp if it doesn't exist
if not Path(REPO).exists():
    print(f"[INFO] Cloning llama.cpp into {REPO}")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", REPO], check=True)
else:
    print(f"[INFO] llama.cpp already exists at {REPO}")

[INFO] Cloning llama.cpp into /content/llama.cpp


In [ ]:
!ls -l /content/llama.cpp/ | grep convert

-rwxr-xr-x  1 root root 451148 Oct 22 00:38 convert_hf_to_gguf.py
-rwxr-xr-x  1 root root  24353 Oct 22 00:38 convert_hf_to_gguf_update.py
-rwxr-xr-x  1 root root  19106 Oct 22 00:38 convert_llama_ggml_to_gguf.py
-rwxr-xr-x  1 root root  20291 Oct 22 00:38 convert_lora_to_gguf.py


# Export tokenizer from the base model

In [ ]:
# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained("NousResearch/llama-2-7b-chat-hf")

# Save it into the merged model folder
tokenizer.save_pretrained("/kaggle/working/llama-2-7b-codeAlpaca-merged")

('/kaggle/working/llama-2-7b-codeAlpaca-merged/tokenizer_config.json',
 '/kaggle/working/llama-2-7b-codeAlpaca-merged/special_tokens_map.json',
 '/kaggle/working/llama-2-7b-codeAlpaca-merged/tokenizer.model',
 '/kaggle/working/llama-2-7b-codeAlpaca-merged/added_tokens.json',
 '/kaggle/working/llama-2-7b-codeAlpaca-merged/tokenizer.json')

# Export into GGUF using Llama CPP

In [ ]:
!cd /content/llama.cpp && python3 /content/llama.cpp/convert_hf_to_gguf.py /kaggle/working/llama-2-7b-codeAlpaca-merged --outfile /content/model-f16.gguf

JAX version 0.5.3, Flax version 0.10.6 available.
INFO:hf-to-gguf:Loading model: llama-2-7b-codeAlpaca-merged
loading configuration file /kaggle/working/llama-2-7b-codeAlpaca-merged/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.55.2",
  "use_cache": true,
  "vocab_size": 32000
}

INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
loading configuration file /k

# Quantize and Export the Model

In [ ]:
# This script runs the llama.cpp quantizer to convert a full-precision GGUF model (F16)
# into a smaller quantized version (Q4), reducing model size and improving inference speed.
#
# Quantization helps deploy large models on lower-resource hardware by lowering
# weight precision (e.g., from 16-bit floats to 4-bit integers).
#
# Common quantization levels:
# - q8_0    → 8-bit (highest accuracy, largest size)
# - q5_K    → 5-bit hybrid (good balance between size & quality)
# - q4_K_M  → 4-bit modern (fastest with good accuracy) (CURRENT)
# - q4_0    → 4-bit legacy (smallest, but least accurate)
#
# This example uses q4_K_M, the most widely used default for llama.cpp models.

# Paths to the input full-precision model and output quantized model
F16_GGUF_PATH = "/content/model-f16.gguf"  # full precision GGUF
Q4_GGUF_PATH = "/content/model-q4_k_m.gguf"  # destination file

# Run quantization using q4_K_M scheme
! /content/llama.cpp/build/bin/llama-quantize {F16_GGUF_PATH} {Q4_GGUF_PATH} q4_K_M

main: build = 1 (03792ad)
main: built with cc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0 for x86_64-linux-gnu
main: quantizing '/content/model-f16.gguf' to '/content/model-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from /content/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 2 7b codeAlpaca Merged
llama_model_loader: - kv   3:                           general.finetune str              = codeAlpaca-merged
llama_model_loader: - kv   4:                           general.basename str              = llama-2
llama_model_loader: - kv   5:                     

# Mount Google Drive and save model with timestamped name

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Define base save directory
drive_save_dir = "/content/drive/MyDrive/models"
os.makedirs(drive_save_dir, exist_ok=True)

# Add timestamp and quantization tag to filename
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
model_base_name = "llama-2-7b-codeAlpaca"
new_model_name = f"{model_base_name}-{successful_qt}-{timestamp}.gguf"

# Full path inside Google Drive
drive_model_path = os.path.join(drive_save_dir, new_model_name)

# Copy quantized GGUF to Drive
shutil.copy(Q4_GGUF_PATH, drive_model_path)

print(f"[INFO] Model saved to Google Drive: {drive_model_path}")

Mounted at /content/drive
[INFO] Model saved to Google Drive: /content/drive/MyDrive/models/llama-2-7b-codeAlpaca-None-2025-10-22_0127.gguf


# User Download

In [ ]:
# # Offer model download in Colab or report file path
# try:
#     from google.colab import files
#     files.download(Q4_GGUF_PATH)
# except Exception:
#     print("[INFO] Non-Colab environment; GGUF at:", Q4_GGUF_PATH)

# Install, configure and check llama-cpp-python

In [ ]:
# Check python version
!python3 --version

Python 3.12.12


In [ ]:
# Remove any CPU wheels, install the exact CUDA 12.4 for the upcoming test integration
!pip -q install -U https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.16-cu124/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

# Show GPU and version (for sanity), then hard-restart the Python runtime so the new wheel is actually loaded
import llama_cpp

print("llama-cpp-python:", llama_cpp.__version__)
print(subprocess.run(["nvidia-smi"], text=True, capture_output=True).stdout)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
llama-cpp-python: 0.3.16
Wed Oct 22 01:31:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             58W /  400W |   53

In [ ]:
# Import llama.cpp core library interface
from llama_cpp import llama_cpp as CLI

# Retrieve system and build information from llama.cpp
info = CLI.llama_print_system_info()

# Decode bytes output if necessary
try:
    info = info.decode()
except Exception:
    pass

# Display system info; ensure CUDA/cuBLAS support is active
# Expect "CUDA = 1" or "cuBLAS = 1" for GPU acceleration
print(info)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A100-SXM4-80GB, compute capability 8.0, VMM: yes


CUDA : ARCHS = 500,520,530,600,610,620,700,720,750,800,860,870,890,900 | FORCE_MMQ = 1 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 


# Inspect LLaMA GGUF Model Metadata and Extracting Transformer Layer Count"

In [ ]:
from llama_cpp import Llama

# Fast: read GGUF metadata only (no tensors loaded)
llm = Llama(model_path="/content/model-q4_k_m.gguf", vocab_only=True, verbose=False)

# Dump all metadata keys (optional)
for k, v in llm.metadata.items():
    print(f"{k} = {v}")

# Grab just the block count (transformer layers)
block_count = int(llm.metadata["llama.block_count"])
print("llama.block_count =", block_count)

llama_context: n_ctx_per_seq (512) > n_ctx_train (0) -- possible training context overflow


llama.attention.head_count_kv = 32
llama.feed_forward_length = 11008
llama.embedding_length = 4096
tokenizer.ggml.add_bos_token = true
general.size_label = 7B
general.type = model
general.file_type = 15
general.finetune = codeAlpaca-merged
llama.context_length = 4096
general.name = Llama 2 7b codeAlpaca Merged
tokenizer.ggml.bos_token_id = 1
general.basename = llama-2
tokenizer.ggml.padding_token_id = 0
llama.rope.freq_base = 10000.000000
general.architecture = llama
llama.block_count = 32
llama.attention.head_count = 32
llama.attention.key_length = 128
llama.attention.value_length = 128
tokenizer.ggml.pre = default
llama.vocab_size = 32000
tokenizer.ggml.model = llama
tokenizer.ggml.add_sep_token = false
general.quantization_version = 2
llama.attention.layer_norm_rms_epsilon = 0.000010
tokenizer.ggml.eos_token_id = 2
tokenizer.ggml.unknown_token_id = 0
tokenizer.ggml.add_eos_token = false
llama.rope.dimension_count = 128
tokenizer.ggml.add_space_prefix = false
llama.block_count = 32


# Check the quantized GGUF model with a sample prompt:

In [ ]:
# Define assistant role and secure review instructions
system_prompt = (
    "You are a senior Python security engineer and tutor. Your job is to review code, identify "
    "vulnerabilities in plain English and provide concise, production-ready fixes. Favor secure "
    "defaults, minimal dependencies and clear comments. Never provide exploit payloads or step-by-step "
    "attack instructions; focus on remediation, safe patterns and brief justifications. When asked to "
    "rewrite code, preserve functionality where possible, improve input validation, add minimal tests, "
    "and document trade-offs."
)

# Vulnerable example demonstrating unsafe pickle deserialization
code_snippet = """import pickle

def load_user_data(serialized_data):
    \"""
    Deserializes user data from a byte stream.
    ...
    \"""
    user_data = pickle.loads(serialized_data)
    return user_data

if __name__ == "__main__":
    malicious_input = input("Enter serialized data: ")
    try:
        serialized_data = bytes.fromhex(malicious_input)
        load_user_data(serialized_data)
    except Exception as e:
        print(f"Deserialization failed: {e}")"""

# Instruct LLM to analyze and securely rewrite code
user_prompt = (
    "Please review the following script and do all of the following in order:\n"
    "1) Identify the primary vulnerability and why it’s dangerous.\n"
    "2) Explain realistic impact and when it can be exploited.\n"
    "3) Rewrite the program securely **without `pickle`** (e.g., use `json` + validation) while keeping the same basic CLI behavior.\n"
    "4) Add a tiny unit test snippet that demonstrates safe behavior (no network/files).\n"
    "5) Briefly list safer serialization options and when to use them.\n"
    "6) Keep the answer concise and do **not** include exploit examples.\n\n"
    f"Here is the code to analyze and refactor:\n\n```python\n{code_snippet}\n```"
)

# Structure messages for chat-style model input
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# Initialize Llama runtime with model and settings
llm = Llama(
    model_path="/content/model-q4_k_m.gguf",
    n_ctx=4096,  # Max context tokens the Llama 2 7B can handle
    n_threads=os.cpu_count(),  # Number of CPU threads for computation
    n_batch=1024,  # Tokens processed per batch (affects speed/memory)
    n_gpu_layers=-1,  # Offload all layers to GPU for faster inference
    chat_format="llama-2",  # Use Llama 2 chat-style prompt formatting
    use_mmap=True,  # Memory-map model file to save RAM
    use_mlock=False,  # Prevent locking model into RAM (saves memory)
    verbose=False,  # Keep the output less polluted
)


# Generate model response using chat completion
resp = llm.create_chat_completion(
    messages=messages,  # Conversation history (system + user roles)
    max_tokens=1200,  # Limit length of generated response (customize later)
    temperature=0.2,  # Controls creativity; lower = more focused
    top_p=0.9,  # Nucleus sampling for balanced diversity
    top_k=40,  # Restrict token selection to top-k choices
    repeat_penalty=1.05,  # Discourage repetitive output patterns
    stop=["</s>", "[/INST]"],  # Stop tokens for chat-style models
)

# Output the model's concise secure review
print(resp["choices"][0]["message"]["content"].strip())

```python
import json

def load_user_data(serialized_data):
    """
    Deserializes user data from a byte stream.
    ...
    """
    try:
        user_data = json.loads(serialized_data)
        return user_data
    except Exception as e:
        print(f"Deserialization failed: {e}")

if __name__ == "__main__":
    malicious_input = input("Enter serialized data: ")
    try:
        serialized_data = bytes.fromhex(malicious_input)
        user_data = load_user_data(serialized_data)
        print(f"User data deserialized successfully: {user_data}")
    except Exception as e:
        print(f"Deserialization failed: {e}")
```

The primary vulnerability is the `pickle` serialization, which can be exploited by sending a malicious `pickle` file to the program. This can lead to code execution or data tampering.

The realistic impact is that an attacker could send a specially crafted `pickle` file to the program, which would execute arbitrary code or modify user data.

To rewrite the program s